# ML-09 — Validation, calibration, and claims

[Open in Colab](https://colab.research.google.com/github/Catacombs666/ML_int-/blob/main/work/notebooks/w06_validation_audit.ipynb)

Built on the FlyRank ML Internship dataset — https://flyrank.ai. This notebook publishes aggregate or derived diagnostics only.

In [1]:
import os, sys, subprocess, importlib.util
from pathlib import Path
candidates=[Path.cwd(), *Path.cwd().parents, Path('/content/ML_int-')]
ROOT=next((p for p in candidates if (p/'data/raw/content_refresh_anonymized.csv').is_file()),None)
if ROOT is None:
    destination=Path('/content/ML_int-') if Path('/content').exists() else Path.cwd()/'ML_int-'
    subprocess.run(['git','clone','https://github.com/Catacombs666/ML_int-.git',str(destination)],check=True)
    ROOT=destination
if os.environ.get('CAPSTONE_OFFLINE') != '1':
    subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(ROOT/'work/requirements.txt')],check=True)
sys.path.insert(0,str(ROOT/'work/scripts'))
import json, pandas as pd, numpy as np
import ctr_capstone as study
print('Setup ready: public starter data; no token required; numerical dependencies loaded.')


Setup ready: public starter data; no token required; numerical dependencies loaded.


## Split and provenance checks

Disjoint clients, known byte-level source fingerprint, explicit starter scope.

In [2]:
receipt_path=ROOT/'work/outputs/ctr_metrics.json'
if not receipt_path.exists():
    study.run()
r=json.loads(receipt_path.read_text())
print(json.dumps(r['splits'],indent=2))
print(json.dumps(r['leakage_checks'],indent=2))
print('Source SHA-256:',r['data']['sha256'])

{
  "train": {
    "rows": 8483,
    "clients": 17
  },
  "validation": {
    "rows": 4330,
    "clients": 5
  },
  "test": {
    "rows": 3913,
    "clients": 6
  }
}
{
  "client_overlap": 0,
  "content_overlap": 0,
  "target_or_identifier_features": 0,
  "preprocessing_fit_on_train_only": true,
  "winner_selected_on_validation_only": true,
  "future_forecast_claim": false
}
Source SHA-256: c43bdac4eccfa17fcd8a33974fe36f2c998c03a3ae3af8d80cf712abab5d6396


## Uncertainty and segments

Client bootstrap is exploratory with only six test clients; inspect where the model loses as well as wins.

In [3]:
print(json.dumps(r['bootstrap'],indent=2))
print(pd.DataFrame(r['subgroup_errors']).to_string(index=False))

{
  "difference_selected_minus_baseline_pp": -0.031597559277413347,
  "client_bootstrap_95pct_interval_pp": [
    -0.044987970369798895,
    -0.021518584760153066
  ],
  "replicates": 2000,
  "clusters": 6,
  "interpretation": "Exploratory client resampling with few clusters; negative favors selected method."
}
  group_field              group    n  selected_mae_pp  baseline_mae_pp
position_band                1–3   67         0.178325         0.323867
position_band               4–10 1477         0.198743         0.209767
position_band              11–20 1209         0.161205         0.219231
position_band              21–50 1096         0.097904         0.113951
position_band                >50   64         0.104700         0.096765
 content_type comparison article   70         0.293889         0.308041
 content_type    keyword article 3828         0.171605         0.203335


## Calibration

Observed and predicted rates by decile expose biased reference gaps.

In [4]:
print(pd.DataFrame(r['calibration']).to_string(index=False))

 decile   n  predicted_ctr_pp  observed_ctr_pp
      1 392          0.076379         0.076551
      2 391          0.134142         0.195079
      3 391          0.160213         0.211416
      4 391          0.189023         0.245816
      5 392          0.218370         0.244401
      6 391          0.247565         0.280548
      7 391          0.269574         0.290455
      8 391          0.298339         0.336543
      9 391          0.344984         0.361935
     10 392          0.464811         0.416440


## Safe conclusion

Observed predictive association on unseen clients; no temporal generalization or causal content uplift.

In [5]:
print('Supported: measured held-out CTR error and reference ranking.')
print('Unsupported: future clicks, Google causal factors, recoverable-click or refresh-uplift claims.')

Supported: measured held-out CTR error and reference ranking.
Unsupported: future clicks, Google causal factors, recoverable-click or refresh-uplift claims.
